In this notebook I'm going to obtain an overview, cleaning and formatting of the dataset before the EDA. 

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [2]:
# # change the format to reduce the size of the dataset

# df = pd.read_csv("../data/raw/ncr_ride_bookings.csv")
# df.to_parquet("../data/raw/rides.parquet", engine='fastparquet', index=False)

In [3]:
df_raw = pd.read_parquet("../data/raw/rides.parquet")

In [4]:
df_raw.head(15)

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,NaN,None,NaN,None,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,NaN,None,NaN,None,NaN,None,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,NaN,None,NaN,None,NaN,None,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,NaN,None,NaN,None,NaN,None,737.0,48.21,4.1,4.3,UPI
5,2024-02-06,09:44:56,"""CNR4096693""",Completed,"""CID4670564""",Auto,AIIMS,Narsinghpur,5.1,18.1,NaN,None,NaN,None,NaN,None,316.0,4.85,4.1,4.6,UPI
6,2024-06-17,15:45:58,"""CNR2002539""",Completed,"""CID6800553""",Go Mini,Vaishali,Punjabi Bagh,7.1,20.4,NaN,None,NaN,None,NaN,None,640.0,41.24,4.0,4.1,UPI
7,2024-03-19,17:37:37,"""CNR6568000""",Completed,"""CID8610436""",Auto,Mayur Vihar,Cyber Hub,12.1,16.5,NaN,None,NaN,None,NaN,None,136.0,6.56,4.4,4.2,UPI
8,2024-09-14,12:49:09,"""CNR4510807""",No Driver Found,"""CID7873618""",Go Sedan,Noida Sector 62,Noida Sector 18,NaN,NaN,NaN,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None
9,2024-12-16,19:06:48,"""CNR7721892""",Incomplete,"""CID5214275""",Auto,Rohini,Adarsh Nagar,6.1,26.0,NaN,None,NaN,None,1.0,Other Issue,135.0,10.36,NaN,NaN,Cash


In [5]:
df_raw.info(memory_usage = "deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  object 
 1   Time                               150000 non-null  object 
 2   Booking ID                         150000 non-null  object 
 3   Booking Status                     150000 non-null  object 
 4   Customer ID                        150000 non-null  object 
 5   Vehicle Type                       150000 non-null  object 
 6   Pickup Location                    150000 non-null  object 
 7   Drop Location                      150000 non-null  object 
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by Customer  1050

To do: 
1. Convert names into snake case
2. Map target variable with the right values
3. Remove quotes from string values in Booking ID and Customer ID
4. Optimize column types 
  - Data and time are object → convert into one datetime column
- Check the nulls?

## Convert names into snake case

In [6]:
import re 
df_columns_clean = df_raw.copy()
def to_snake_case(df):
    new_columns = []
    for col in df.columns:
        col = col.strip()
        col = re.sub(r'[\s\-]+', '_', col)
        col = re.sub(r'[^\w_]', '', col)
        col = col.lower()
        new_columns.append(col)
    
    df.columns = new_columns
    return df

df_columns_clean = to_snake_case(df_columns_clean)
print(df_columns_clean.columns.tolist())

['date', 'time', 'booking_id', 'booking_status', 'customer_id', 'vehicle_type', 'pickup_location', 'drop_location', 'avg_vtat', 'avg_ctat', 'cancelled_rides_by_customer', 'reason_for_cancelling_by_customer', 'cancelled_rides_by_driver', 'driver_cancellation_reason', 'incomplete_rides', 'incomplete_rides_reason', 'booking_value', 'ride_distance', 'driver_ratings', 'customer_rating', 'payment_method']


## Map target variable

In [7]:
df_target_mapped = df_columns_clean.copy()

booking_mapping = {
    "Completed": 0,
    "Cancelled by Driver": 1,
    "No Driver Found": 1,
    "Cancelled by Customer": 1,
    "Incomplete": 0
}

df_target_mapped['cancelled'] = df_target_mapped['booking_status'].map(booking_mapping)
del df_target_mapped["booking_status"]
df_target_mapped.head()

,date,time,booking_id,customer_id,vehicle_type,pickup_location,drop_location,avg_vtat,avg_ctat,cancelled_rides_by_customer,reason_for_cancelling_by_customer,cancelled_rides_by_driver,driver_cancellation_reason,incomplete_rides,incomplete_rides_reason,booking_value,ride_distance,driver_ratings,customer_rating,payment_method,cancelled
0,2024-03-23,12:29:38,"""CNR5884300""","""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None,1
1,2024-11-29,18:01:39,"""CNR1326809""","""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,NaN,None,NaN,None,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,0
2,2024-08-23,08:56:10,"""CNR8494506""","""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,NaN,None,NaN,None,NaN,None,627.0,13.58,4.9,4.9,Debit Card,0
3,2024-10-21,17:17:25,"""CNR8906825""","""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,NaN,None,NaN,None,NaN,None,416.0,34.02,4.6,5.0,UPI,0
4,2024-09-16,22:08:00,"""CNR1950162""","""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,NaN,None,NaN,None,NaN,None,737.0,48.21,4.1,4.3,UPI,0


## Remove quotes from string values in Booking ID and Customer ID

In [8]:
df_clean_quotes = df_target_mapped.copy()

cols_with_quotes = ["booking_id", 
                    "customer_id"]

for col in cols_with_quotes:
    df_clean_quotes[col] = df_clean_quotes[col].str.strip('"')

df_clean_quotes.head()

,date,time,booking_id,customer_id,vehicle_type,pickup_location,drop_location,avg_vtat,avg_ctat,cancelled_rides_by_customer,reason_for_cancelling_by_customer,cancelled_rides_by_driver,driver_cancellation_reason,incomplete_rides,incomplete_rides_reason,booking_value,ride_distance,driver_ratings,customer_rating,payment_method,cancelled
0,2024-03-23,12:29:38,CNR5884300,CID1982111,eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None,1
1,2024-11-29,18:01:39,CNR1326809,CID4604802,Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,NaN,None,NaN,None,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,0
2,2024-08-23,08:56:10,CNR8494506,CID9202816,Auto,Khandsa,Malviya Nagar,13.4,25.8,NaN,None,NaN,None,NaN,None,627.0,13.58,4.9,4.9,Debit Card,0
3,2024-10-21,17:17:25,CNR8906825,CID2610914,Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,NaN,None,NaN,None,NaN,None,416.0,34.02,4.6,5.0,UPI,0
4,2024-09-16,22:08:00,CNR1950162,CID9933542,Bike,Ghitorni Village,Khan Market,5.3,19.6,NaN,None,NaN,None,NaN,None,737.0,48.21,4.1,4.3,UPI,0


## Optimize column types

In [18]:
df_typed = df_clean_quotes.copy()

df_typed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   date                               150000 non-null  object 
 1   time                               150000 non-null  object 
 2   booking_id                         150000 non-null  object 
 3   customer_id                        150000 non-null  object 
 4   vehicle_type                       150000 non-null  object 
 5   pickup_location                    150000 non-null  object 
 6   drop_location                      150000 non-null  object 
 7   avg_vtat                           139500 non-null  float64
 8   avg_ctat                           102000 non-null  float64
 9   cancelled_rides_by_customer        10500 non-null   float64
 10  reason_for_cancelling_by_customer  10500 non-null   object 
 11  cancelled_rides_by_driver          2700

### Temporal features

Create column `datetime` and convert it along`date` to datetime. Keep `time` as object because datetime needs a date (or a dummy date) to be converted

In [ ]:
df_typed["datetime"] = pd.to_datetime(df_typed["date"].astype(str) + " " + df_typed["time"].astype(str))
df_typed["date"] = pd.to_datetime(df_typed["date"])

df_typed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 22 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   date                               150000 non-null  object        
 1   time                               150000 non-null  object        
 2   booking_id                         150000 non-null  object        
 3   customer_id                        150000 non-null  object        
 4   vehicle_type                       150000 non-null  object        
 5   pickup_location                    150000 non-null  object        
 6   drop_location                      150000 non-null  object        
 7   avg_vtat                           139500 non-null  float64       
 8   avg_ctat                           102000 non-null  float64       
 9   cancelled_rides_by_customer        10500 non-null   float64       
 10  reason_for_cancellin

### Categorical
1. Columns with "object" data type and no nulls are directly turned into Category
   
2. Columns with "object" data type and nulls are turned into Category based on threshold. Threshold is set to 0.5, that means less than 50% of values are unique. If threshold is less that this value, the column will be turned into string[pyarrow] type. This type os more efficient than "object" and it's ideal for parquet



In [21]:
cat_ambiguous = ["reason_for_cancelling_by_customer",
                 "driver_cancellation_reason",
                 "incomplete_rides_reason"]
cat_threshold = 0.5

for col in cat_ambiguous:
    if col in df_typed.columns:
        num_unique = df_typed[col].nunique(dropna = False)
        ratio_unique = num_unique/len(df_typed)

        if ratio_unique <= cat_threshold:
            df_typed[col] = df_typed[col].astype("category")
            print(f"- Column {col} has been transformed into Category")
            print(f"Ratio of the column: {ratio_unique:.4%}")
        else:
            df_typed[col] = df_typed[col].astype("string[pyarrow]")
            print(f"- Column {col} has been transformed into string[pyarrow]")
            print(f"Ratio of the column: {ratio_unique:.4%}")


- Column reason_for_cancelling_by_customer has been transformed into Category
Ratio of the column: 0.0040%
- Column driver_cancellation_reason has been transformed into Category
Ratio of the column: 0.0033%
- Column incomplete_rides_reason has been transformed into Category
Ratio of the column: 0.0027%


In [22]:
cat_columns = ["booking_id", 
               "customer_id", 
               "vehicle_type", 
               "pickup_location", 
               "drop_location",
               "payment_method"]

for col in cat_columns:
    if col in df_typed.columns:
        df_typed[col] = df_typed[col].astype("category")
        print(f"- Column {col} has been transformed into Category")

- Column booking_id has been transformed into Category
- Column customer_id has been transformed into Category
- Column vehicle_type has been transformed into Category
- Column pickup_location has been transformed into Category
- Column drop_location has been transformed into Category
- Column payment_method has been transformed into Category


### Numerical

In [ ]:
#TODO - calculate size saved with data type optimization and change format 

In [ ]:
float_columns = ["avg_vtat", 
                 "avg_ctat", 
                 "cancelled_rides_by_customer",
                 "cancelled_rides_by_driver",
                 "incomplete_rides",
                 "booking_value",
                 "ride_distance",
                 "driver_ratings",
                 "customer_rating" ]

int_columns = ["cancelled"]